# Task 03 — Spark SQL and Window Functions

## Setup

In [ ]:
import os, shutil, subprocess, sys

def _find_java():
    """Check if java is available on PATH or in JAVA_HOME."""
    if os.environ.get("JAVA_HOME"):
        java_bin = os.path.join(os.environ["JAVA_HOME"], "bin", "java")
        if os.path.isfile(java_bin):
            return True
    return shutil.which("java") is not None

def _find_installed_jdk():
    """Look for a previously installed JDK in ~/.jdk."""
    jdk_dir = os.path.expanduser("~/.jdk")
    if os.path.exists(jdk_dir):
        for d in sorted(os.listdir(jdk_dir)):
            candidate = os.path.join(jdk_dir, d)
            if os.path.isdir(candidate) and os.path.isfile(os.path.join(candidate, "bin", "java")):
                return candidate
    return None

# Auto-install Java if not available (required by PySpark)
if not _find_java():
    prev = _find_installed_jdk()
    if prev:
        os.environ["JAVA_HOME"] = prev
        print(f"Using JAVA_HOME={prev}")
    else:
        print("Java not found. Installing JDK 17 via install-jdk...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "install-jdk"])
        import jdk
        path = jdk.install("17")
        os.environ["JAVA_HOME"] = path
        print(f"JAVA_HOME set to {path}")
else:
    print(f"Java found. JAVA_HOME={os.environ.get('JAVA_HOME', '(system)')}")

In [ ]:
import os
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StringType

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Task03") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

FIXTURES = os.path.abspath(os.path.join("fixtures", "input"))
if not os.path.exists(FIXTURES):
    FIXTURES = os.path.abspath(os.path.join("..", "fixtures", "input"))

emp_df = spark.read.csv(os.path.join(FIXTURES, "employees.csv"), header=True, inferSchema=True)
orders_df = spark.read.csv(os.path.join(FIXTURES, "orders.csv"), header=True, inferSchema=True)
customers_df = spark.read.csv(os.path.join(FIXTURES, "customers.csv"), header=True, inferSchema=True)

emp_df.createOrReplaceTempView("employees")
orders_df.createOrReplaceTempView("orders")
customers_df.createOrReplaceTempView("customers")
print("Setup complete. Views registered: employees, orders, customers")

## Task 3.1: SQL — Department stats

Write a SQL query that returns `department`, `cnt` (count of employees), `avg_sal` (average salary rounded to 0).
Order by `avg_sal` descending. Store result in `dept_sql`.

In [ ]:
# YOUR CODE HERE
dept_sql = spark.sql("""

""")

# TEST — Do not modify
rows = {r["department"]: r for r in dept_sql.collect()}
assert len(rows) == 3
assert rows["Engineering"]["cnt"] == 6
assert rows["Engineering"]["avg_sal"] == round(sum([95000,88000,102000,91000,98000,105000])/6)
print("Task 3.1 passed!")

## Task 3.2: SQL with JOIN — Customer order summary

Write a SQL query joining orders and customers to get:
`name` (customer), `city`, `total_orders`, `total_amount`.
Order by `total_amount` DESC. Store in `cust_sql`.

In [ ]:
# YOUR CODE HERE
cust_sql = spark.sql("""

""")

# TEST — Do not modify
assert cust_sql.count() == 5
rows = {r["name"]: r for r in cust_sql.collect()}
assert rows["Alice"]["total_orders"] == 4
assert rows["Alice"]["total_amount"] == 2750.0
print("Task 3.2 passed!")

## Task 3.3: Window — Salary rank within department

Using DataFrame API (not SQL), add a `dept_rank` column to `emp_df` ranking employees by salary within each department (highest = rank 1).
Store result in `emp_ranked`. Keep all original columns plus `dept_rank`.

In [ ]:
# YOUR CODE HERE
emp_ranked = ...

# TEST — Do not modify
assert "dept_rank" in emp_ranked.columns
noah = emp_ranked.filter(F.col("name") == "Noah").collect()[0]
assert noah["dept_rank"] == 1  # highest in Engineering
alice = emp_ranked.filter(F.col("name") == "Alice").collect()[0]
assert alice["dept_rank"] == 4  # 4th in Engineering
print("Task 3.3 passed!")

## Task 3.4: Window — Running total per customer

Add a `running_total` column to orders showing cumulative sum of `amount` per `customer_id`, ordered by `order_date`.
Store in `orders_running`. Keep all original columns plus `running_total`.

In [ ]:
# YOUR CODE HERE
orders_running = ...

# TEST — Do not modify
cust1 = orders_running.filter(F.col("customer_id") == 1).orderBy("order_date").collect()
assert cust1[0]["running_total"] == 1200.0  # first order
assert cust1[1]["running_total"] == 1700.0  # 1200 + 500
assert cust1[2]["running_total"] == 1850.0  # + 150
assert cust1[3]["running_total"] == 2750.0  # + 900
print("Task 3.4 passed!")

## Task 3.5: UDF — Categorize orders

Create a UDF `size_category` that takes an amount and returns:
- "Small" if amount < 300
- "Medium" if 300 <= amount < 1000
- "Large" if amount >= 1000

Apply it to create `orders_categorized` with a new `size` column.

In [ ]:
# YOUR CODE HERE
orders_categorized = ...

# TEST — Do not modify
rows = orders_categorized.select("order_id", "amount", "size").collect()
sizes = {r["order_id"]: r["size"] for r in rows}
assert sizes[101] == "Large"    # 1200
assert sizes[102] == "Medium"   # 800
assert sizes[105] == "Small"    # 150
print("Task 3.5 passed!")

## Task 3.6: SQL CTE — Employees above department average

Write a SQL query using a CTE to find employees whose salary is above their department's average.
Return: `name`, `department`, `salary`, `dept_avg` (rounded to 0).
Order by `salary` DESC. Store in `above_avg`.

In [ ]:
# YOUR CODE HERE
above_avg = spark.sql("""

""")

# TEST — Do not modify
names = {r["name"] for r in above_avg.collect()}
assert "Noah" in names    # 105k, eng avg ~96.5k
assert "Eve" in names     # 102k
assert "Karen" in names   # 98k
assert "Diana" in names   # 78k, mkt avg ~73.5k
assert "Leo" in names     # 75k
assert "Jack" in names    # 73k, sales avg ~69.8k
assert "Grace" in names   # 71k
assert "Charlie" not in names  # 72k < 73.5k
print("Task 3.6 passed!")

## Cleanup

In [ ]:
spark.stop()
print("All tasks done!")